# 01 - Data Exploration

## Objectif
Explorer et comprendre les données de prix des actions du Dow Jones 30.

**Points clés :**
- Téléchargement des données via Yahoo Finance
- Vérification de la qualité des données
- Analyse exploratoire (distributions, corrélations)
- Identification des périodes de stress

In [ ]:
# Configuration
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Modules du projet
from src.data_loader import load_universe_data, load_benchmark_data, compute_returns
from src.config import get_config

config = get_config()
print(f"Univers: {len(config.DOW_JONES_30)} actions du Dow Jones 30")
print(f"Période: {config.START_DATE} à {config.END_DATE}")

## 1. Chargement des données

Nous utilisons les prix ajustés (splits et dividendes inclus) de Yahoo Finance.
Les données sont rééchantillonnées en fréquence mensuelle.

In [ ]:
# Téléchargement des données (avec cache)
prices = load_universe_data(use_cache=True, verbose=True)
benchmark = load_benchmark_data(use_cache=True, verbose=True)

print(f"\nShape des prix: {prices.shape}")
print(f"Première date: {prices.index[0]}")
print(f"Dernière date: {prices.index[-1]}")

In [ ]:
# Aperçu des données
prices.head()

## 2. Qualité des données

Vérification des valeurs manquantes et anomalies.

In [ ]:
# Valeurs manquantes
missing = prices.isna().sum()
print("Valeurs manquantes par action:")
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante")

In [ ]:
# Première date disponible pour chaque action
first_valid = prices.apply(lambda x: x.first_valid_index())
print("\nPremière date disponible par action:")
print(first_valid.sort_values())

In [ ]:
# Visualisation de la disponibilité des données
fig, ax = plt.subplots(figsize=(14, 8))

availability = prices.notna().astype(int)
sns.heatmap(availability.T, cmap='Greens', cbar_kws={'label': 'Data Available'})

# Réduire le nombre de labels sur l'axe x
n_ticks = 10
tick_positions = np.linspace(0, len(prices)-1, n_ticks, dtype=int)
ax.set_xticks(tick_positions)
ax.set_xticklabels([prices.index[i].strftime('%Y-%m') for i in tick_positions], rotation=45)

ax.set_title('Disponibilité des données par action')
ax.set_xlabel('Date')
ax.set_ylabel('Action')
plt.tight_layout()
plt.show()

## 3. Évolution des prix

Visualisation de l'évolution des prix normalisés (base 100).

In [ ]:
# Normalisation des prix (base 100)
prices_normalized = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 8))

# Tracer toutes les actions en gris clair
prices_normalized.plot(ax=ax, alpha=0.3, color='gray', legend=False)

# Benchmark en noir épais
benchmark_normalized = benchmark / benchmark.iloc[0] * 100
benchmark_normalized.plot(ax=ax, color='black', linewidth=2, label='S&P 500')

# Top 5 et Bottom 5 performers
total_returns = prices.iloc[-1] / prices.iloc[0] - 1
top5 = total_returns.nlargest(5).index
bottom5 = total_returns.nsmallest(5).index

for ticker in top5:
    ax.plot(prices_normalized[ticker], linewidth=1.5, label=f"{ticker} (+{total_returns[ticker]:.0%})")

ax.axhline(100, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Évolution des prix (Base 100)')
ax.set_xlabel('Date')
ax.set_ylabel('Prix normalisé')
ax.legend(loc='upper left', fontsize=9)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 4. Distribution des rendements

Analyse des caractéristiques statistiques des rendements mensuels.

In [ ]:
# Calcul des rendements mensuels
returns = compute_returns(prices)

# Statistiques descriptives
stats = returns.describe().T
stats['skewness'] = returns.skew()
stats['kurtosis'] = returns.kurtosis()
stats['sharpe'] = stats['mean'] / stats['std'] * np.sqrt(12)  # Annualisé

print("Statistiques des rendements mensuels:")
stats[['mean', 'std', 'min', 'max', 'skewness', 'kurtosis', 'sharpe']].round(4)

In [ ]:
# Distribution des rendements (ensemble de l'univers)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
all_returns = returns.values.flatten()
all_returns = all_returns[~np.isnan(all_returns)]

axes[0].hist(all_returns, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(0, color='red', linestyle='--', label='Zero')
axes[0].axvline(np.mean(all_returns), color='green', linestyle='--', label=f'Mean: {np.mean(all_returns):.2%}')
axes[0].set_title('Distribution des rendements mensuels')
axes[0].set_xlabel('Rendement mensuel')
axes[0].set_ylabel('Densité')
axes[0].legend()

# QQ-plot
from scipy import stats as scipy_stats
scipy_stats.probplot(all_returns, dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot vs Distribution Normale')

plt.tight_layout()
plt.show()

print(f"\nSkewness moyenne: {returns.skew().mean():.3f}")
print(f"Kurtosis moyenne: {returns.kurtosis().mean():.3f}")
print("Note: Kurtosis > 0 indique des queues épaisses (fat tails)")

## 5. Corrélations

Analyse des corrélations entre actions.

In [ ]:
# Matrice de corrélation
corr_matrix = returns.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlBu_r', 
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Matrice de corrélation des rendements')
plt.tight_layout()
plt.show()

# Corrélation moyenne
upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
avg_corr = upper_tri.stack().mean()
print(f"\nCorrélation moyenne entre actions: {avg_corr:.3f}")

In [ ]:
# Évolution de la corrélation dans le temps (rolling)
# Corrélation moyenne glissante sur 12 mois
rolling_corr = returns.rolling(12).corr()

# Moyenne des corrélations par date
def mean_corr_per_date(df):
    result = []
    for date in df.index.get_level_values(0).unique():
        corr = df.loc[date]
        upper = corr.where(np.triu(np.ones_like(corr, dtype=bool), k=1))
        result.append({'date': date, 'avg_corr': upper.stack().mean()})
    return pd.DataFrame(result).set_index('date')

avg_rolling_corr = mean_corr_per_date(rolling_corr)

fig, ax = plt.subplots(figsize=(14, 5))
avg_rolling_corr['avg_corr'].plot(ax=ax)
ax.axhline(avg_corr, color='red', linestyle='--', label=f'Moyenne: {avg_corr:.3f}')
ax.fill_between(avg_rolling_corr.index, 0, avg_rolling_corr['avg_corr'], alpha=0.3)

# Marquer les crises
ax.axvspan('2008-09-01', '2009-03-01', alpha=0.2, color='red', label='Crise 2008')
ax.axvspan('2020-02-01', '2020-04-01', alpha=0.2, color='orange', label='COVID-19')

ax.set_title('Corrélation moyenne glissante (12 mois)')
ax.set_ylabel('Corrélation moyenne')
ax.legend()
plt.tight_layout()
plt.show()

print("Note: Les corrélations augmentent pendant les crises (diversification réduite)")

## 6. Périodes de stress

Identification et analyse des drawdowns majeurs.

In [ ]:
# Drawdown du benchmark
benchmark_cum = (1 + benchmark.pct_change()).cumprod()
benchmark_drawdown = benchmark_cum / benchmark_cum.cummax() - 1

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Prix
benchmark_cum.plot(ax=axes[0], color='steelblue')
axes[0].set_title('S&P 500 - Performance cumulée')
axes[0].set_ylabel('Valeur ($1 initial)')

# Drawdown
benchmark_drawdown.iloc[:, 0].plot(ax=axes[1], color='red', alpha=0.7)
axes[1].fill_between(benchmark_drawdown.index, 0, benchmark_drawdown.iloc[:, 0], alpha=0.3, color='red')
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')
axes[1].axhline(-0.2, color='orange', linestyle='--', label='-20% (Bear market)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMax Drawdown: {benchmark_drawdown.min().values[0]:.2%}")

In [ ]:
# Performance pendant les crises
crisis_periods = {
    'Crise financière 2008': ('2008-09-01', '2009-03-31'),
    'COVID-19 2020': ('2020-02-01', '2020-03-31'),
    'Bear market 2022': ('2022-01-01', '2022-10-31'),
}

print("Performance pendant les crises:")
print("=" * 60)

for crisis_name, (start, end) in crisis_periods.items():
    crisis_returns = returns.loc[start:end]
    benchmark_crisis = benchmark.pct_change().loc[start:end]
    
    cum_return = (1 + crisis_returns).prod() - 1
    benchmark_cum_ret = (1 + benchmark_crisis).prod().values[0] - 1
    
    print(f"\n{crisis_name}:")
    print(f"  S&P 500: {benchmark_cum_ret:.2%}")
    print(f"  DJ30 moyenne: {cum_return.mean():.2%}")
    print(f"  Meilleure action: {cum_return.idxmax()} ({cum_return.max():.2%})")
    print(f"  Pire action: {cum_return.idxmin()} ({cum_return.min():.2%})")

## 7. Résumé

**Observations clés:**

1. **Qualité des données**: Les données sont complètes pour la plupart des actions depuis 2005

2. **Distribution des rendements**: 
   - Légère asymétrie négative (skewness < 0)
   - Queues épaisses (kurtosis > 0) - événements extrêmes plus fréquents que prévu par une loi normale

3. **Corrélations**:
   - Corrélation moyenne ~0.4-0.5
   - Augmentation des corrélations pendant les crises (réduction de la diversification)

4. **Périodes de stress**:
   - 2008: Crise financière (drawdown ~50%)
   - 2020: COVID-19 (drawdown rapide puis récupération)
   - 2022: Bear market taux (drawdown ~25%)

**Implications pour la stratégie:**
- Le momentum cross-sectionnel exploite la dispersion des rendements
- Un filtre de régime peut aider à réduire l'exposition pendant les crises
- Les coûts de transaction doivent être intégrés (turnover significatif)

In [ ]:
# Sauvegarde des statistiques clés
summary_stats = pd.DataFrame({
    'Metric': ['Nombre d\'actions', 'Période (mois)', 'Rendement mensuel moyen', 
               'Volatilité mensuelle', 'Corrélation moyenne', 'Max Drawdown (benchmark)'],
    'Value': [len(prices.columns), len(prices), f"{returns.mean().mean():.2%}",
              f"{returns.std().mean():.2%}", f"{avg_corr:.3f}", 
              f"{benchmark_drawdown.min().values[0]:.2%}"]
})
print("\nRésumé des données:")
print(summary_stats.to_string(index=False))